# Notebook 06 - Clinical plausibility study: Objective (c), RQ3/H3
Thesis Section 3.5.4 and **Equation 4**: Δ = mean rating of real maps − mean rating of control maps.

**Part A (before the study):** picks 60 test cases (12 per superclass), makes a real and a control map for each, shuffles the 120 maps, and creates (1) a PDF booklet with one map per page, (2) an Excel rating form per rater, and (3) a secret key file. Do this only after HREC approval.

**Part B (after the study):** reads the returned forms and computes Δ with its 95% CI, inter-rater agreement and the Jaccard region overlap.

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

In [ ]:
from preprocessing.ptbxl import SUPERCLASSES, LEADS
from training.train import load_trained, predict
from xai.shap_explainer import explain_shap
from xai.gradcam import explain_gradcam
from evaluation.plausibility import make_rating_packet, rating_difference, inter_rater_agreement, jaccard_overlap
from xai.segment_alignment import align_batch, SEGMENTS
P = f"{DATA}/processed/ptbxl100"
X_test, y_test = np.load(f"{P}/X_test.npy"), np.load(f"{P}/y_test.npy")
masks = np.load(f"{RESULTS}/segment_masks_test.npy")
store = dict(np.load(f"{RESULTS}/grids_test.npz"))
STUDY = f"{RESULTS}/clinician_study"; os.makedirs(STUDY, exist_ok=True)
# Which explanation to show the cardiologists: choose the method that did best in notebook 05.
ARCH, METHOD = "cnn1d", "shap"
RATERS = ["Rater_A", "Rater_B", "Rater_C"]      # use codes, never names, in these files
model = load_trained(ARCH, 5, f"{RESULTS}/models/ptbxl_{ARCH}/seed0")[0]

## Part A, Step 6.1 - Choose the 60 cases and make real and control maps
Cases are test records the model classifies correctly for that superclass. The **control** map for a case is the map of a different case (shuffled), so it is a realistic-looking but wrong explanation.

In [ ]:
rng = np.random.default_rng(2026)
probs = predict(model, X_test)
th = load_trained(ARCH, 5, f"{RESULTS}/models/ptbxl_{ARCH}/seed0")[1]
per_class = 2 if QUICK_TEST else 12
cases = []
for c, cls in enumerate(SUPERCLASSES):
    ok = np.where((y_test[:, c] == 1) & (probs[:, c] >= th[c]))[0]
    ok = np.setdiff1d(ok, [x["row"] for x in cases])
    for r in rng.choice(ok, min(per_class, len(ok)), replace=False):
        cases.append({"case_id": f"C{len(cases) + 1:02d}", "row": int(r), "class": cls, "c": c})
cases = pd.DataFrame(cases)
bg = np.load(f"{P}/X_train.npy", mmap_mode="r")[:100]
real = np.zeros((len(cases), 12, 1000), dtype=np.float32)
for c in cases.c.unique():
    sel = (cases.c == c).to_numpy()
    rows = cases.row[sel].to_numpy()
    if METHOD == "shap":
        real[sel] = explain_shap(model, X_test[rows], np.asarray(bg), int(c), nsamples=10 if QUICK_TEST else 100)
    else:
        real[sel] = np.repeat(explain_gradcam(model, X_test[rows], int(c))[:, None], 12, axis=1)
perm = np.roll(np.arange(len(cases)), 1 + rng.integers(len(cases) - 1))    # every case gets another case's map
control = real[perm]
np.savez_compressed(f"{STUDY}/maps.npz", real=real, control=control, perm=perm)
cases["region_overlap_case"] = False
cases.loc[rng.choice(len(cases), len(cases) // 2, replace=False), "region_overlap_case"] = True   # 30 of 60
cases.to_csv(f"{STUDY}/cases.csv", index=False)
cases.head()

## Step 6.2 - Shuffle and blind: booklet, rating forms and the secret key

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
sheet, key = make_rating_packet(cases.case_id.tolist(), seed=7)
key.to_csv(f"{STUDY}/SECRET_key_do_not_send.csv", index=False)
t = np.arange(1000) / 100
def draw(page_title, x, sv):
    lim = np.abs(sv).max() + 1e-12
    fig, axes = plt.subplots(12, 1, figsize=(8.27, 11.69), sharex=True)
    for l in range(12):
        ax = axes[l]
        ax.imshow(np.abs(sv[l])[None, :], aspect="auto", cmap="Reds", vmin=0, vmax=lim,
                  extent=[0, 10, x[l].min(), x[l].max()], alpha=0.7)
        ax.plot(t, x[l], "k", lw=0.7); ax.set_yticks([]); ax.set_ylabel(LEADS[l], rotation=0, labelpad=12)
    axes[-1].set_xlabel("time (s)"); fig.suptitle(page_title, fontsize=11)
    plt.tight_layout(); return fig
with PdfPages(f"{STUDY}/rating_booklet.pdf") as pdf:
    for _, r in key.iterrows():
        k = int(np.where(cases.case_id == r.case_id)[0][0])
        cs = cases.iloc[k]
        sv = real[k] if r.map_type == "real" else control[k]
        fig = draw(f"{r.map_id}  |  Case {cs.case_id}  |  Model prediction: {cs['class']}\n"
                   "Darker red = the model relied more on this part of the ECG", X_test[cs.row], sv)
        pdf.savefig(fig); plt.close(fig)
print("Booklet pages:", len(key))

In [ ]:
# One Excel form per rater: Instructions, Ratings, and the Region checklist (30 cases)
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, Alignment
def make_form(rater):
    wb = Workbook()
    ws = wb.active; ws.title = "Instructions"
    lines = [f"ECG explanation rating form - {rater}",
             "1. Open rating_booklet.pdf. Each page shows a 12-lead ECG, the model's predicted class, and a map (red shading).",
             "2. For each map, score how well the red areas highlight the ECG features relevant to the predicted class:",
             "   1 = not at all, 2 = poorly, 3 = partly, 4 = mostly, 5 = completely.",
             "3. Answer whether this explanation would be useful in Nigerian clinical practice (Yes/No).",
             "4. In the 'Regions' sheet, for the listed cases only, enter 1 in each lead x segment cell you consider clinically relevant.",
             "5. Rate each page independently. Do not discuss cases with other raters.",
             "Some maps are deliberately incorrect; you are not told which."]
    for i, l in enumerate(lines, 1):
        ws.cell(i, 1, l).font = Font(bold=(i == 1))
    ws.column_dimensions["A"].width = 120
    ws2 = wb.create_sheet("Ratings")
    hdr = ["map_id", "case_id", "rating_1_to_5", "useful_in_nigerian_practice_yes_no", "comments"]
    ws2.append(hdr)
    for _, r in sheet.iterrows():
        ws2.append([r.map_id, r.case_id, None, None, None])
    dv1 = DataValidation(type="whole", operator="between", formula1="1", formula2="5", allow_blank=True)
    dv2 = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
    ws2.add_data_validation(dv1); ws2.add_data_validation(dv2)
    dv1.add(f"C2:C{len(sheet) + 1}"); dv2.add(f"D2:D{len(sheet) + 1}")
    for col, w in zip("ABCDE", [10, 10, 16, 34, 50]):
        ws2.column_dimensions[col].width = w
    ws3 = wb.create_sheet("Regions")
    ws3.append(["case_id", "segment"] + LEADS)
    for cid in cases.case_id[cases.region_overlap_case]:
        for s in SEGMENTS:
            ws3.append([cid, s] + [None] * 12)
    wb.save(f"{STUDY}/rating_form_{rater}.xlsx")
for r in RATERS:
    make_form(r)
print(sorted(os.listdir(STUDY)))

**Send each rater:** `rating_booklet.pdf` and their own `rating_form_Rater_X.xlsx`. **Never send** `SECRET_key_do_not_send.csv`. Pilot the form with one clinician first (Section 3.5.4).

## Part B, Step 6.3 - Read the returned forms
Put the completed forms back in the `clinician_study` folder with the same names. In a quick test, made-up ratings are generated so you can see the analysis run.

In [ ]:
key = pd.read_csv(f"{STUDY}/SECRET_key_do_not_send.csv")
if QUICK_TEST:     # simulated ratings, ONLY to test the code
    sim = np.random.default_rng(0)
    long = pd.concat([key.assign(rater=r, rating=np.clip(np.where(key.map_type == "real", 4, 2.5)
                                                         + sim.normal(0, 1, len(key)), 1, 5).round(),
                                 useful=np.where(sim.random(len(key)) < 0.5, "Yes", "No")) for r in RATERS])
else:
    parts = []
    for r in RATERS:
        f = pd.read_excel(f"{STUDY}/rating_form_{r}.xlsx", sheet_name="Ratings")
        f = f.rename(columns={"rating_1_to_5": "rating", "useful_in_nigerian_practice_yes_no": "useful"})
        parts.append(f.merge(key, on=["map_id", "case_id"]).assign(rater=r))
    long = pd.concat(parts)
long = long.dropna(subset=["rating"])
print(len(long), "ratings from", long.rater.nunique(), "raters")
long.to_csv(f"{STUDY}/ratings_long.csv", index=False)

## Step 6.4 - Table 4.12: Δ (Equation 4), agreement and usefulness

In [ ]:
d = rating_difference(long)
agr = inter_rater_agreement(long)
rows = [{"Measure": "Mean rating, real maps", "Value": round(long[long.map_type == "real"].rating.mean(), 2)},
        {"Measure": "Mean rating, control maps", "Value": round(long[long.map_type == "control"].rating.mean(), 2)},
        {"Measure": "Δ = real − control (Eq. 4), 95% CI", "Value": f"{d['delta']:.2f} ({d['ci_low']:.2f} to {d['ci_high']:.2f})"},
        {"Measure": "Wilcoxon signed-rank p (per case)", "Value": round(d["wilcoxon_p"], 4)},
        {"Measure": agr["measure"], "Value": round(agr["value"], 3)}]
for mt in ["real", "control"]:
    u = (long[long.map_type == mt].useful.astype(str).str.lower() == "yes").mean()
    rows.append({"Measure": f"Judged useful in Nigerian practice ({mt} maps)", "Value": f"{100 * u:.0f}%"})
table412 = pd.DataFrame(rows)
table412.to_csv(f"{RESULTS}/tables/table4_12_plausibility.csv", index=False); table412

In [ ]:
# Figure 4.7: ratings of real vs control maps
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
long.boxplot(column="rating", by="map_type", ax=ax[0]); ax[0].set_title("Ratings by map type"); ax[0].set_xlabel("")
long[long.map_type == "real"].merge(pd.read_csv(f"{STUDY}/cases.csv")[["case_id", "class"]]) \
    .groupby("class").rating.mean().reindex(SUPERCLASSES).plot.bar(ax=ax[1], color="#1f6f8b")
ax[1].set_title("Mean rating of real maps by class"); plt.suptitle("")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_7_ratings.png", dpi=300); plt.show()

## Step 6.5 - Region overlap (Jaccard) on the 30 checklist cases

In [ ]:
cases = pd.read_csv(f"{STUDY}/cases.csv")
maps = np.load(f"{STUDY}/maps.npz")
model_grids = align_batch(maps["real"], masks[cases.row.to_numpy()])
jac = []
for r in RATERS:
    if QUICK_TEST:
        reg = None
    else:
        reg = pd.read_excel(f"{STUDY}/rating_form_{r}.xlsx", sheet_name="Regions").fillna(0)
    for k, cs in cases[cases.region_overlap_case].iterrows():
        if reg is None:
            marks = (np.random.default_rng(k).random((5, 12)) < 0.15).astype(int)
        else:
            marks = reg[reg.case_id == cs.case_id].set_index("segment").reindex(SEGMENTS)[LEADS].to_numpy()
        if marks.sum() > 0:
            jac.append({"rater": r, "case_id": cs.case_id, "class": cs["class"],
                        "jaccard": jaccard_overlap(marks, model_grids[k])})
jac = pd.DataFrame(jac)
print("Mean Jaccard (model top-k vs rater):", round(jac.jaccard.mean(), 3))
jac.groupby("class").jaccard.mean().round(3)

**Done.** For Chapter 4, Section 4.6 keep Table 4.12, Figure 4.7 and the Jaccard results. Remember these results are exploratory because of the small rater pool. Next: notebook 07 (EchoNext).